# 📘 Notebook 16: Exploring the Data with Spark

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 4 of 6 in this module · (16 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Read CSV files into Spark, and say what `inferSchema` costs and when to write the schema yourself
- Join a star schema into one **wide table** in Spark, and read the **join strategy** in the query plan
- Ask the same question with the **DataFrame API** and with **Spark SQL**
- Carry out an **exploratory data analysis**: distributions, missing values, outliers and relationships
- Keep the heavy work in Spark and bring only **small results** back to pandas for plotting

> **Prerequisites:** Notebooks 09 and 10 (Spark DataFrames, Spark SQL, query plans). Notebook 14 (this warehouse). Plotting with Matplotlib, from *From Python to Deep Learning and Fine-Tuning*, Notebook 07.
>
> 🔭 **Why this notebook matters:** *exploratory data analysis* (EDA) is the conversation an analyst has with a data set before trusting it with a decision: how is each variable distributed, what is missing, what is strange, what moves together. On a laptop it is done in pandas. When the data lives in a cluster, the same conversation has to be held through Spark, and one rule becomes essential: let Spark compute over all the rows, and bring back to your own machine only what fits on a chart. This notebook holds that conversation with the AdventureWorks sales.

> 📦 **Libraries used in this notebook.**
>
> - **PySpark** (Notebook 09), in local mode: the cores of this machine play the part of a cluster.
> - **pandas** and **Matplotlib**, for the small results and the charts.
>
> The cell below installs PySpark. In Google Colab, PySpark and the Java it needs are already present and the cell finishes in a moment.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
%pip install -q pyspark

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # the same Python for the driver and the workers

## 1. Starting Spark

In [ ]:
spark = (SparkSession.builder
         .master("local[*]")
         .appName("adventureworks")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "with", spark.sparkContext.defaultParallelism, "cores")

## 2. Reading the tables, and the cost of guessing types

### Intuition first
A CSV file is only text. Spark can **guess** each column's type (`inferSchema`), but to guess it must read the whole file once before reading it again for real. On a few megabytes that is nothing. On a terabyte it doubles the cost of every load, and a wrong guess (a postal code read as a number, losing its leading zero) can corrupt the data silently. So production pipelines **declare** the schema. Let us measure the difference on the largest table.

In [ ]:
path = f"{DATA}/FactInternetSales.csv"

started = time.perf_counter()
guessed = spark.read.csv(path, header=True, inferSchema=True)
guessed.count()
t_guess = time.perf_counter() - started

# Declare the schema: take the column names, and say which are integers, decimals and text.
names = guessed.columns
integers = {"ProductKey", "OrderDateKey", "DueDateKey", "ShipDateKey", "CustomerKey", "PromotionKey", "CurrencyKey",
            "SalesTerritoryKey", "SalesOrderLineNumber", "RevisionNumber", "OrderQuantity"}
texts = {"SalesOrderNumber", "CarrierTrackingNumber", "CustomerPONumber", "OrderDate", "DueDate", "ShipDate"}
schema = T.StructType([T.StructField(n, T.IntegerType() if n in integers else
                                        T.StringType() if n in texts else T.DoubleType()) for n in names])
started = time.perf_counter()
declared = spark.read.csv(path, header=True, schema=schema)
declared.count()
t_declared = time.perf_counter() - started
print(f"inferSchema: {t_guess:.2f} s   declared schema: {t_declared:.2f} s")

The declared read is faster, because the file is read once (part of the first time is also Spark starting up). For the small dimension tables we let Spark guess; for the fact table we keep the declared schema.

In [ ]:
def read(name):
    return spark.read.csv(f"{DATA}/{name}.csv", header=True, inferSchema=True)

sales = (declared
         .withColumn("OrderDate", F.to_date(F.substring("OrderDate", 1, 10)))
         .withColumn("Year", F.year("OrderDate"))
         .withColumn("Month", F.month("OrderDate")))
product, subcategory, category = read("DimProduct"), read("DimProductSubcategory"), read("DimProductCategory")
customer, geography = read("DimCustomer"), read("DimGeography")
sales.select("SalesOrderNumber", "OrderDate", "Year", "Month", "SalesAmount").show(3)

## 3. One wide table

### Intuition first
For exploration, analysts prefer one wide table to a star: every row a sale, every column something about it. Building it takes five joins. On a cluster a join is the expensive operation of Notebook 10, because matching rows may live on different machines and must be **shuffled** across the network. Unless one side is small: then Spark copies the small table to every machine, a **broadcast join**, and nothing else moves. All our dimensions are small.

In [ ]:
prod = (product.select("ProductKey", "EnglishProductName", "ProductSubcategoryKey")
        .join(subcategory.select("ProductSubcategoryKey", "EnglishProductSubcategoryName", "ProductCategoryKey"),
              "ProductSubcategoryKey", "left")
        .join(category.select("ProductCategoryKey", "EnglishProductCategoryName"), "ProductCategoryKey", "left"))
cust = (customer.select("CustomerKey", "GeographyKey", "BirthDate", "Gender", "MaritalStatus", "YearlyIncome",
                        "TotalChildren", "EnglishEducation", "EnglishOccupation", "HouseOwnerFlag",
                        "NumberCarsOwned", "CommuteDistance")
        .join(geography.select("GeographyKey", "EnglishCountryRegionName"), "GeographyKey", "left"))

wide = (sales.join(prod, "ProductKey", "left").join(cust, "CustomerKey", "left")
        .withColumnRenamed("EnglishProductCategoryName", "Category")
        .withColumnRenamed("EnglishProductSubcategoryName", "Subcategory")
        .withColumnRenamed("EnglishCountryRegionName", "Country")
        .withColumn("Margin", F.col("SalesAmount") - F.col("TotalProductCost")))
wide.cache()
print(wide.count(), "rows,", len(wide.columns), "columns")

In [ ]:
plan = wide._jdf.queryExecution().executedPlan().toString()
final, _, initial = plan.partition("== Initial Plan ==")
for label, part in [("plan before running", initial), ("plan actually run", final)]:
    print(f"{label:<20} broadcast joins: {part.count('BroadcastHashJoin')}   sort-merge joins: {part.count('SortMergeJoin')}")

Before running, Spark planned a couple of the joins as **sort-merge** joins, the expensive kind that shuffles both sides, because it could only estimate the size of the joined tables. While running, it measured them, found them small, and switched every join to a **broadcast**. This re-planning during execution is called **adaptive query execution**, and it is switched on by default since Spark 3.2. We also called `cache()`, because every question below reads this table again, and without the cache Spark would redo the joins each time (Notebook 09).

## 4. One question, two languages

Revenue by year and category, first with the DataFrame API and then with SQL. Both compile to the same plan.

In [ ]:
(wide.groupBy("Year", "Category")
     .agg(F.count("*").alias("lines"), F.round(F.sum("SalesAmount"), 0).alias("revenue"))
     .orderBy("Year", F.desc("revenue"))
     .show())

In [ ]:
wide.createOrReplaceTempView("sales_wide")
spark.sql('''
    SELECT Country, ROUND(SUM(SalesAmount)) AS revenue, COUNT(DISTINCT CustomerKey) AS customers,
           ROUND(100 * SUM(Margin) / SUM(SalesAmount), 1) AS margin_pct
    FROM sales_wide
    GROUP BY Country
    ORDER BY revenue DESC
''').show()

## 5. Exploratory data analysis

EDA asks four questions of every variable that matters: how is it **distributed**, what is **missing**, what is **unusual**, and what is it **related** to. Spark answers each over all rows; we bring back only the answer.

### 5.1 Summary statistics and missing values

In [ ]:
wide.select("SalesAmount", "Margin", "Freight", "YearlyIncome", "TotalChildren").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

In [ ]:
missing = wide.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in wide.columns]).toPandas().T[0]
missing[missing > 0]

Only the two columns that Notebook 13 found entirely empty. The result of a whole-table check comes back as a single row: that is the pattern of this notebook.

### 5.2 The unit of analysis

The order line is the right unit for questions about products. For questions about **people** the unit is the customer, so we aggregate to one row per customer: total spend, number of orders, age, and days since the last order. That table has 18,484 rows, small enough to bring back to pandas.

In [ ]:
last_day = sales.agg(F.max("OrderDate")).first()[0]
customers = (wide.groupBy("CustomerKey", "Gender", "MaritalStatus", "YearlyIncome", "EnglishEducation", "Country",
                          "BirthDate", "TotalChildren", "NumberCarsOwned", "HouseOwnerFlag", "CommuteDistance")
             .agg(F.round(F.sum("SalesAmount"), 2).alias("spend"),
                  F.countDistinct("SalesOrderNumber").alias("orders"),
                  F.max("OrderDate").alias("last_order"))
             .withColumn("age", F.floor(F.months_between(F.lit(last_day), F.to_date("BirthDate")) / 12))
             .withColumn("days_since_last", F.datediff(F.lit(last_day), "last_order")))
cpd = customers.toPandas()
print(cpd.shape)
cpd[["spend", "orders", "age", "days_since_last"]].describe().round(1)

### 5.3 Distributions

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
cpd["spend"].plot.hist(bins=60, ax=ax[0], title="Spend per customer")
np.log10(cpd["spend"]).plot.hist(bins=60, ax=ax[1], title="Spend per customer, log10 scale")
cpd["age"].plot.hist(bins=30, ax=ax[2], title="Age at the last order date")
plt.tight_layout()
plt.show()

The spend is **heavily skewed**: most customers spent a little, a few spent a great deal. On a logarithmic scale the picture separates into two groups, people who bought accessories or clothing and people who bought a bicycle. For a skewed variable the **median** is a more honest summary than the mean, and any model that uses it will probably work better with its logarithm.

In [ ]:
print(f"mean spend {cpd['spend'].mean():.0f}   median spend {cpd['spend'].median():.0f}")

### 5.4 What is unusual?

The **interquartile range** rule marks values far beyond the middle half of the data: above Q3 + 1.5 × (Q3 − Q1). Spark can compute quantiles on the full data with `approxQuantile`, which trades a tiny, controllable error for not having to sort everything.

In [ ]:
q1, q3 = customers.approxQuantile("spend", [0.25, 0.75], 0.001)
upper = q3 + 1.5 * (q3 - q1)
flagged = customers.filter(F.col("spend") > upper).count()
print(f"Q1 = {q1:.0f}   Q3 = {q3:.0f}   upper fence = {upper:.0f}   customers above it: {flagged}")

An outlier is a question, not an error. These are the company's best customers. Removing them would make the averages look tidier and the analysis wrong. The decision, keep, cap or study separately, depends on the purpose, and it should always be written down.

### 5.5 Relationships

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
cpd.boxplot(column="spend", by="EnglishEducation", ax=ax[0], showfliers=False)
ax[0].set_title("Spend by education (outliers hidden)")
ax[0].tick_params(axis="x", rotation=25)
cpd.groupby("YearlyIncome")["spend"].median().plot(marker="o", ax=ax[1], title="Median spend by yearly income")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
cpd[["spend", "orders", "age", "YearlyIncome", "TotalChildren", "NumberCarsOwned", "days_since_last"]].corr().round(2)

The correlations are weak, the largest being between spend and number of orders, which is close to arithmetic. Income is mildly related to spend, and age hardly at all. Weak correlations are a finding too: they warn that no single attribute will predict spending well, which matters for the next notebook.

### 5.6 Change over time

Bring back one row per month and category, a few dozen rows, and plot them.

In [ ]:
monthly = (wide.groupBy(F.date_trunc("month", "OrderDate").alias("month"), "Category")
               .agg(F.sum("SalesAmount").alias("revenue"))
               .toPandas()
               .pivot(index="month", columns="Category", values="revenue"))
monthly.plot(figsize=(12, 4), title="Revenue per month and category", logy=True)
plt.show()

On a logarithmic axis the three businesses are visible side by side: bicycles throughout, accessories and clothing from the end of 2012, and the last, incomplete month falling off the cliff. The fall is not a crisis; it is the edge of the data, and an analysis that forgot it would report one.

## 6. Where we stand

| Step of the EDA | How it was done in Spark | What came back |
|---|---|---|
| Load with the right types | a declared schema for the big table | nothing |
| Combine the star | broadcast joins into one cached wide table | nothing |
| Summaries, missing values | `summary()`, a null count per column | one small table each |
| Distributions, relationships | aggregate to customers, then pandas | 18,484 rows |
| Outliers | `approxQuantile` on all rows | three numbers |
| Trends | aggregate by month and category | a few dozen rows |

The pattern is always the same: compute over everything where the data lives, and move only answers. The next notebook keeps the same customers and asks Spark to learn from them.

---
## ✏️ Exercises

### Exercise 1 (Margins by subcategory)
With Spark SQL on `sales_wide`, list the ten subcategories with the highest **margin percentage**, together with their revenue. Do the most profitable subcategories bring much revenue?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
spark.sql('''
    SELECT Subcategory, ROUND(SUM(SalesAmount)) AS revenue,
           ROUND(100 * SUM(Margin) / SUM(SalesAmount), 1) AS margin_pct
    FROM sales_wide GROUP BY Subcategory ORDER BY margin_pct DESC LIMIT 10
''').show()
```

*Ten subcategories share exactly the same margin, 62.6 percent: in this sample data the cost of many accessories was evidently set as a fixed share of the price. A value repeated that exactly is worth noticing in any data set, because real data is rarely so tidy. Their revenue is small next to the bicycles: a margin ranking and a revenue ranking answer different business questions.*

</details>

### Exercise 2 (Repeat customers by country)
Using `customers` (the Spark DataFrame), compute for each country the share of customers who placed **more than one** order. Return a small pandas table sorted by that share.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
(customers.groupBy("Country")
          .agg(F.round(100 * F.avg((F.col("orders") > 1).cast("int")), 1).alias("repeat_pct"),
               F.count("*").alias("customers"))
          .orderBy(F.desc("repeat_pct"))
          .toPandas())
```

*The average of a 0/1 column is a share. Computing it in Spark and returning six rows is the pattern of the whole notebook.*

</details>

### Exercise 3 (How approximate is approximate?)
Compute the median spend three ways: `approxQuantile` with relative error 0.05, with 0.001, and exactly in pandas (`cpd['spend'].median()`). Time the two Spark calls.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for err in (0.05, 0.001):
    started = time.perf_counter()
    m = customers.approxQuantile("spend", [0.5], err)[0]
    print(f"error {err}: median {m:.2f}  in {time.perf_counter() - started:.2f} s")
print(f"exact (pandas): {cpd['spend'].median():.2f}")
```

*The relative error is an error in **rank**: with 0.05 the answer may be any value between the 45th and the 55th percentile. Spend is so skewed around its middle that this moves the median far, here by roughly a third. With 0.001 the answer is close. On billions of rows an exact median needs a full sort across the cluster, so the approximate one is the practical choice, but choose the error with the distribution in mind.*

</details>

### Exercise 4 (Plot without collecting, a little harder)
A histogram of `SalesAmount` over all **order lines** would need every value. Instead, let Spark count the lines in 20 equal-width bins of `log10(SalesAmount)` and bring back only the 20 counts. Plot them as a bar chart.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
lo, hi = wide.select(F.min(F.log10("SalesAmount")), F.max(F.log10("SalesAmount"))).first()
width = (hi - lo) / 20
bins = (wide.select(F.least(F.floor((F.log10("SalesAmount") - lo) / width), F.lit(19)).alias("bin"))
            .groupBy("bin").count().orderBy("bin").toPandas())
bins["from"] = (lo + bins["bin"] * width).round(2)
bins.plot.bar(x="from", y="count", figsize=(10, 3), legend=False, title="Order lines by log10(SalesAmount)")
plt.show()
```

*Twenty numbers travel instead of sixty thousand, or sixty billion. This is how dashboards on large data draw their histograms.*

</details>

## 🔑 Key takeaways

- Declare the **schema** of large files: `inferSchema` reads the data twice and can guess wrong.
- Joining a big table to small ones uses **broadcast joins**, which move almost nothing; `cache()` a table you will read many times.
- The DataFrame API and Spark SQL are two languages for the same plans.
- EDA asks about **distributions**, **missing values**, **outliers** and **relationships**; choose the right **unit of analysis** first.
- Skewed variables call for the **median** and often a **logarithm**; outliers are questions, not errors.
- Compute where the data lives and bring back only **answers**: aggregates, quantiles, binned counts.

**Next:** *Notebook 17: Machine Learning with Spark MLlib*, where the customer table becomes the input of models trained by Spark.

In [ ]:
spark.stop()

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*